# 상품 정보 추출과 요약 — 데이터 생성 파이프라인

아마존 상품 설명에서 정보를 뽑아, 구매자에게 맞는 요약문을 만들어 봅니다.
오픈소스 LLM으로 **데이터 생성 파이프라인**을 한 단계씩 쌓아 가는 실습입니다.

## 무엇을 하게 되나

1. 모델에게 상품에 대해 **무엇을 아는지** 묻습니다
2. 그 답을 다음 단계의 입력으로 넘기며, 작은 단계 7개를 **사슬로 잇습니다**
3. 마지막에 전체 상품에 돌려 **학습 데이터**를 만듭니다 — 내일 SFT가 이걸 씁니다

왜 큰 작업을 LLM 하나에 통째로 맡기지 않고 작게 쪼개는지는,
파이프라인을 만들어 가면서 직접 확인하게 됩니다.



In [3]:
# vLLM 은 별도 venv 에서 서버로 띄운다 (verify/02_vllm_venv.sh 참조).
# openai<3 : llama-index-llms-openai 가 openai<3 을 요구한다(3일차와 같은 환경)
%pip install -q -U 'openai<3' datasets pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 24.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [1]:
import json
from datasets import load_dataset
from openai import OpenAI

from pydantic import BaseModel, Field
from typing import Annotated, List

먼저 사용할 아마존 데이터를 불러옵니다.

In [2]:
data = load_dataset('Taekyoon/test_amazon', split='train[:10]')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


amazon_ec_examples.json:   0%|          | 0.00/190k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [4]:
data

Dataset({
    features: ['text'],
    num_rows: 10
})

In [3]:
print(data[0]['text'])


### Product Catalog Information

#### Product Information

- Product Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET
- Brand: F, a, t,  , S, h, a, r, k
- Categories: Electronics > Television & Video > Video Glasses
- Product Attributes: 
Date First Available: August 2, 2014
Manufacturer: Fatshark

- Product Description: 

Description:
Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.



모델 서버(vLLM)가 떠 있는지 먼저 확인합니다.
확인 후 강사 안내가 있을 때까지 다음 코드는 실행하지 말아 주세요.

In [5]:
# 별도 터미널에서 실행한다:
#   source /opt/vllm-env/bin/activate
#   nohup vllm serve Qwen/Qwen3-4B-Instruct-2507 --port 8000 > vllm.log 2>&1 &
!curl -s http://localhost:8000/v1/models || echo "서버가 아직 준비되지 않았습니다"

nohup: appending output to 'nohup.out'


In [ ]:
# 모델이 잘 실행되는지 테스트 해봅니다.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
model_name = "Qwen/Qwen3-4B-Instruct-2507"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(model=model_name,
                                      prompt="정보 이론에 대해서 설명해주세요.")
print("Completion result:", completion.choices[0].text)

### 1단계 — 모델이 무엇을 아는지 묻는다

- 무엇을 뽑을지 우리가 정하기 전에, **모델이 이 상품에서 무엇을 알아볼 수 있는지**부터 묻습니다.
- 원하는 것을 바로 시키는 것보다 모델이 아는 것을 먼저 묻는 쪽이 문제 해결에 도움이 됩니다.

In [ ]:
STEP_1_PROMPT = """목표:
주어진 상품 설명을 분석해서, 이 상품에 어떤 종류의 속성이 있는지 찾아내세요.

할 일:
1. 속성의 종류를 나열합니다 (예: 색상, 크기, 재질 등).
2. 각 속성이 무엇을 뜻하는지 짧게 설명합니다.

주의할 점:

1. 명시된 것만: 상품 설명에 실제로 나온 속성만 포함하세요. 추측하지 마세요.
2. 값이 아니라 종류: 설명에는 구체적인 값(예: "빨강")을 넣지 말고 종류만 설명하세요.
3. JSON 형식: 문법이 올바른 JSON 으로 출력하세요.
4. **모든 출력은 한국어로 작성하세요.** 입력이 영어여도 한국어로 답하세요.

출력 예시:

[
    {"feature_type": "색상", "descript": "상품이 제공하는 색상."},
    {"feature_type": "크기", "descript": "상품의 치수나 규격."},
    {"feature_type": "재질", "descript": "상품을 만든 소재."},
    {"feature_type": "브랜드", "descript": "제조사 또는 브랜드 이름."},
    {"feature_type": "출시연도", "descript": "상품이 출시된 연도."},
]
"""

### 출력을 JSON으로 **강제**합니다

바로 아래 셀에 이 실습 전체를 관통하는 패턴이 처음 나옵니다.

```
pydantic 모델  →  model_json_schema()  →  response_format
```

프롬프트로 "JSON으로 답해줘"라고 **부탁**할 수도 있습니다. 대체로 따릅니다.
그런데 10건 중 1건이 어기면, 그 1건이 다음 단계를 통째로 무너뜨립니다.
7단계가 사슬로 엮여 있으니까요.

**스키마를 주면 다른 형태가 나올 수 없습니다.** 생성 단계에서 문법으로 막습니다.

### 길이 상한을 반드시 겁니다

`Field(max_length=...)`가 붙어 있는 것을 보세요. **이게 없으면 실제로 깨집니다.**

문자열 필드에 상한이 없으면 문법상 무한히 길어질 수 있습니다. 모델이 늘어지다가
`max_tokens`에 걸려 JSON이 **중간에서 잘리고**, 몇 셀 뒤에서
`Unterminated string`으로 죽습니다. 원인에서 가장 먼 곳에서 드러나는 셈입니다.

> 이 실습을 한국어로 바꾸면서 실제로 겪은 일입니다. 영어일 때는 우연히 넘어갔는데
> 출력이 길어지면서 드러났습니다. **"영어에서 됐으니 한국어도 된다"가 성립하지 않습니다.**

### 실패를 삼키지 않습니다

각 함수 끝에 `finish_reason == "length"` 검사가 있습니다.
**잘린 응답은 에러가 아니라 정상 응답**이라 `try/except`에 걸리지 않습니다.
여기서 잡지 않으면 원인에서 멀리 떨어진 곳에서 에러가 납니다.

In [ ]:
# 길이 상한이 없으면 제약 디코딩이 문자열을 끝없이 늘릴 수 있고,
# 그러면 max_tokens 에서 잘려 JSON 자체가 깨진다 (실행 검증에서 확인).
class FeatureType(BaseModel):
    feature_type: str = Field(max_length=40)
    descript: str = Field(max_length=200)

class FeatureTypeList(BaseModel):
    feature_type_list: List[FeatureType] = Field(max_length=20)


feature_type_schema = FeatureTypeList.model_json_schema()


def gen_text_step_1(text):
    try:
        messages = []

        messages.append({"role": "user", "content": STEP_1_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name,
              messages=messages,
              temperature=0.2,   # 추출 작업이다. 높으면 문자열이 늘어진다
              max_tokens=2048,   # 원본은 상한이 아예 없었다
              top_p=0.95,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "feature_type_list", "schema": feature_type_schema}},
            )

        # 길이 상한에 걸리면 JSON 이 중간에서 끊긴다. 그런데 API 는 **성공으로**
        # 응답하므로 아래 except 에 걸리지 않고, 몇 셀 뒤에서 json.loads 가
        # "Unterminated string" 으로 죽는다. 원인에서 먼 곳에서 드러나는 것이 가장 나쁘다.
        if completion.choices[0].finish_reason == "length":
            print("[잘림] 출력이 max_tokens 에 걸렸습니다. "
                  "스키마의 max_length 나 max_tokens 를 늘려야 합니다.")
            return 'error'
        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
feature_type_list = gen_text_step_1(data[0]['text'])
print(feature_type_list)

### 2단계 — 정보를 그룹으로 묶는다

- 항목이 많으면 **그룹으로 묶어** 사람이 보기 쉬운 구조로 정리해 둡니다.
- 이 그룹이 뒤 단계(5·6단계)에서 요약의 뼈대가 됩니다.

In [ ]:
STEP_2_PROMPT = """목표:
앞에서 찾은 속성들을 비슷한 것끼리 묶어서 분류하세요.

할 일:

- 그룹으로 묶기: 속성 종류들을 의미가 비슷한 것끼리 몇 개의 그룹으로 나눕니다.
- JSON 출력: 각 그룹이 속성 목록을 갖는 JSON 형태로 정리합니다.

주의할 점:

- 명시된 것만: 앞에서 나온 속성만 사용하세요. 새로 만들지 마세요.
- 그룹 이름은 짧고 명확하게: 무엇을 묶은 그룹인지 알 수 있어야 합니다.
- JSON 형식: 문법이 올바른 JSON 으로 출력하세요.
- **모든 출력은 한국어로 작성하세요.**

출력 예시:

[
    {
        "subsection": "제품 사양",
        "features": ["색상", "크기", "재질"]
    },
    {
        "subsection": "제품 식별 정보",
        "features": ["브랜드", "출시연도"]
    }
]
"""

In [ ]:
class Subsection(BaseModel):
    subsection: str = Field(max_length=40)
    features: List[Annotated[str, Field(max_length=60)]] = Field(max_length=20)

class SubsectionList(BaseModel):
    subsection_list: List[Subsection] = Field(max_length=10)


subsectoin_schema = SubsectionList.model_json_schema()


def gen_text_step_2(text):
    try:
        messages = []

        messages.append({"role": "user", "content": STEP_2_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name,
              messages=messages,
              max_tokens=2048,
              temperature=0.2,
              top_p=0.95,
              seed=777,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "subsection_list", "schema": subsectoin_schema}},
            )

        # 길이 상한에 걸리면 JSON 이 중간에서 끊긴다. 그런데 API 는 **성공으로**
        # 응답하므로 아래 except 에 걸리지 않고, 몇 셀 뒤에서 json.loads 가
        # "Unterminated string" 으로 죽는다. 원인에서 먼 곳에서 드러나는 것이 가장 나쁘다.
        if completion.choices[0].finish_reason == "length":
            print("[잘림] 출력이 max_tokens 에 걸렸습니다. "
                  "스키마의 max_length 나 max_tokens 를 늘려야 합니다.")
            return 'error'
        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
subsection_list = gen_text_step_2(feature_type_list)
print(subsection_list)

### 3단계 — 실제 값을 뽑는다

- 이제 속성의 **실제 값**을 뽑습니다.
- 추출할 대상을 명시적으로 나열해 주면 결과가 안정됩니다 — 아래에서 1단계의 답을 그 목록으로 씁니다.

In [ ]:
STEP_3_PROMPT = """목표:
주어진 속성 목록을 이용해, 원문에서 실제 값을 뽑아내세요.

할 일:
1. 원문에서 각 속성에 해당하는 값을 찾습니다.
2. 찾은 값을 JSON 형식으로 정리합니다.

주의할 점:
1. 원문에 없는 값은 만들어내지 마세요. 없으면 그 속성은 빼세요.
2. 지정된 JSON 구조를 그대로 지키세요.
3. **모든 출력은 한국어로 작성하세요.** 단, 브랜드명이나 모델명 같은 고유명사는 원문 그대로 두세요.

출력 예시:

[
    {"feature_type": "색상", "value": "노란색"},
    {"feature_type": "브랜드", "value": "APPLE"},
    {"feature_type": "출시연도", "value": "2000"},
]
"""

In [ ]:
class ExtractedFeature(BaseModel):
    feature_type: str = Field(max_length=40)
    value: str = Field(max_length=200)

class ExtractedFeatureList(BaseModel):
    feature_list: List[ExtractedFeature] = Field(max_length=30)


extracted_feature_schema = ExtractedFeatureList.model_json_schema()


def gen_text_step_3(text):
    try:
        messages = []

        messages.append({"role": "user", "content": STEP_3_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name,
              messages=messages,
              max_tokens=2048,
              temperature=0.2,
              top_p=0.95,
              seed=0,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "extracted_features", "schema": extracted_feature_schema}},
            )

        # 길이 상한에 걸리면 JSON 이 중간에서 끊긴다. 그런데 API 는 **성공으로**
        # 응답하므로 아래 except 에 걸리지 않고, 몇 셀 뒤에서 json.loads 가
        # "Unterminated string" 으로 죽는다. 원인에서 먼 곳에서 드러나는 것이 가장 나쁘다.
        if completion.choices[0].finish_reason == "length":
            print("[잘림] 출력이 max_tokens 에 걸렸습니다. "
                  "스키마의 max_length 나 max_tokens 를 늘려야 합니다.")
            return 'error'
        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

### 단계를 사슬로 잇습니다 — 이 실습의 방법론

앞서 Step 1에서 뽑은 **속성 종류 목록**을 Step 3의 프롬프트에 **같이 넣습니다.**

```
Step 1:  "이 상품에 어떤 속성이 있나?"     →  [색상, 용량, 무게, ...]
                                                    │
Step 3:  "이 속성들의 **값**을 뽑아라"  ←──────────┘
         + 원문
```

한 번에 "상품 정보를 전부 뽑아줘"라고 하면 모델이 무엇을 뽑을지 스스로 정합니다.
상품마다 다른 것을 뽑고, 형식도 제각각이 됩니다.

**나눠서 물으면** 각 단계가 단순해지고 결과가 안정됩니다.
그리고 중간 산출물을 눈으로 확인할 수 있어서, **어디서 틀렸는지** 알 수 있습니다.

> 이것이 이 실습의 방법론입니다. 큰 작업을 LLM 하나에 통째로 맡기지 않고,
> **작은 단계로 쪼개 사슬로 잇습니다.** 대신 호출 횟수가 늘어 비용이 커집니다.

In [ ]:
print(feature_type_list)

In [ ]:
extracted_feature_list = gen_text_step_3("Provided Features:\n\n"+ feature_type_list + "\n\n" + data[0]['text'])
features = json.loads(extracted_feature_list)
features

### 4단계 — 구매자 유형을 묻는다

- 이 상품을 살 만한 **구매자 유형**을 모델에게 묻습니다.
- 원래는 사람이 리뷰와 시장을 보고 얻는 인사이트지만, 여기서는 그 초안을 모델에게 맡깁니다.
  **실험적 용도**라는 것은 기억해 두세요.

In [ ]:
STEP_4_PROMPT = """목표:
이 상품을 살 만한 사람들을 사용 목적에 따라 분류하세요.
3~4개 그룹으로 제한합니다.

할 일:
1. 이 상품을 구매할 만한 사용자 유형을 구분합니다.
2. 각 유형을 JSON 형식으로 정리합니다.

주의할 점:
1. 상품 특성에 비추어 말이 되는 그룹으로 나누세요.
2. 각 그룹 설명은 짧고 명확하게 쓰세요.
3. **모든 출력은 한국어로 작성하세요.**
4. 아래 JSON 구조를 그대로 지키세요:

[
  {"user_category": "category name", "describe": "explanation of this category"},
  {"user_category": "category name", "describe": "explanation of this category"},
  ...
]
"""

In [ ]:
# ★ 여기가 실제로 깨진 곳이다. user_category 에 상한이 없어서
#   제약 디코딩이 그 문자열을 쓰다가 max_tokens 에 걸렸다.
class ConsumerCategory(BaseModel):
    user_category: str = Field(max_length=40)
    describe: str = Field(max_length=200)

class ConsumerCategoryList(BaseModel):
    # 프롬프트가 3~4개라고 했으니 스키마로도 못박는다
    consuber_categories: List[ConsumerCategory] = Field(max_length=6)

consumer_category_schema = ConsumerCategoryList.model_json_schema()

def gen_text_step_4(text):
    try:
        messages = []
        messages.append({"role": "user", "content": STEP_4_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name, messages=messages,
              max_tokens=1024, temperature=0.2, top_p=0.95, seed=1234,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "consumer_categories", "schema": consumer_category_schema}},
            )

        # 길이 상한에 걸리면 JSON 이 중간에서 끊긴다. 그런데 API 는 **성공으로**
        # 응답하므로 아래 except 에 걸리지 않고, 몇 셀 뒤에서 json.loads 가
        # "Unterminated string" 으로 죽는다. 원인에서 먼 곳에서 드러나는 것이 가장 나쁘다.
        if completion.choices[0].finish_reason == "length":
            print("[잘림] 출력이 max_tokens 에 걸렸습니다. "
                  "스키마의 max_length 나 max_tokens 를 늘려야 합니다.")
            return 'error'
        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
user_category_list = gen_text_step_4(data[0]['text'])
print(user_category_list)

### 5단계 — 뽑은 정보를 정리한다

- 앞서 뽑은 값들을 그룹(subsection)별로 다시 묶어, 요약문을 만들 재료로 정리합니다.

### 여기는 LLM을 쓰지 않습니다

아래 두 셀을 보세요. **LLM 호출이 없습니다.** 순수 파이썬입니다.

앞에서 뽑은 속성값(Step 3)을 앞에서 만든 그룹(Step 2)에 맞춰 **재배치**하는 일입니다.
이건 규칙이 정해져 있어서 코드로 하면 됩니다.

**모든 것을 LLM에 시키지 않는 것**도 설계입니다.

| | LLM에 맡길 일 | 코드로 할 일 |
|---|---|---|
| 성격 | 판단·해석이 필요 | 규칙이 정해져 있다 |
| 예 | "이 속성은 어느 그룹인가" | 그룹에 맞춰 재배치 |
| 비용 | 호출 시간 · 돈 | 거의 0 |
| 신뢰도 | 흔들린다 | **항상 같다** |

LLM을 한 번 덜 부르면 그만큼 빨라지고 싸지고, **결과가 흔들리지 않습니다.**
파이프라인을 설계할 때 매번 물어야 하는 질문입니다 — **"이건 꼭 LLM 이어야 하나?"**

In [ ]:
# Feature list를 dict 형태로 바꾸기
feature_map = {row['feature_type']: row['value']
               for row in features['feature_list']}

In [ ]:
feature_map

In [ ]:
# 각 Feature를 Subsection에 맞게 정리하기
subsection_dict = {}

subsection_list = json.loads(subsection_list)

for row in subsection_list['subsection_list']:
    subsection = row['subsection']

    subsection_dict[subsection] = {f: feature_map[f]
                                   for f in row['features']
                                   if f in feature_map}

In [ ]:
subsection_dict

### 6단계 — 속성 그룹별로 요약한다

- 구매자 이야기를 붙이기 전에, **제품 정보만으로** 먼저 문장을 만들어 둡니다.

In [ ]:
FACTUAL_SUMMARY_PROMPT = """JSON 형식의 상품 정보가 주어집니다.
이를 사실 위주의 한 줄 문장으로 압축하세요.

아래 형식을 지키세요.
{"summary": "<핵심 특징 한마디>: <한 줄 상세 요약>"}

**모든 출력은 한국어로 작성하세요.**
JSON 형식으로만 출력하세요."""

In [ ]:
class SummaryDescription(BaseModel):
    summary: str = Field(max_length=400)   # 한 줄 요약이다

summary_schema = SummaryDescription.model_json_schema()

def gen_text_summary(text):
    try:
        messages = []
        messages.append({"role": "user", "content": FACTUAL_SUMMARY_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name, messages=messages,
              max_tokens=1024, temperature=0.2, top_p=0.95, seed=1234,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "summary", "schema": summary_schema}},
            )

        # 길이 상한에 걸리면 JSON 이 중간에서 끊긴다. 그런데 API 는 **성공으로**
        # 응답하므로 아래 except 에 걸리지 않고, 몇 셀 뒤에서 json.loads 가
        # "Unterminated string" 으로 죽는다. 원인에서 먼 곳에서 드러나는 것이 가장 나쁘다.
        if completion.choices[0].finish_reason == "length":
            print("[잘림] 출력이 max_tokens 에 걸렸습니다. "
                  "스키마의 max_length 나 max_tokens 를 늘려야 합니다.")
            return 'error'
        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
# dict 객체로 된 Subsection을 다시 텍스트로 표현합니다.
subsection_list = []

for k, v in subsection_dict.items():
    subsection_list.append((k, json.dumps(v, indent=4)))

In [ ]:
subsection_list

In [ ]:
subsection_summary = {}

for row in subsection_list:
    summary_text = gen_text_summary(row[0] + '\n\n' + row[1])
    subsection_summary[row[0]] = summary_text

In [ ]:
subsection_summary

### 7단계 — 구매자 유형별로 요약한다

- 구매자 유형별 관심사를 앞의 제품 요약과 합쳐 **최종 요약문**을 만듭니다.

In [ ]:
FEATURED_SUMMARY_PROMPT = """주어진 정보를 바탕으로 상품 특징 요약문을 여러 줄로 작성하세요.
요약 문장만 출력하고 다른 설명은 붙이지 마세요.

**모든 출력은 한국어로 작성하세요.**
JSON 형식으로만 출력하세요."""

In [ ]:
class SummaryDescription(BaseModel):
    summary: str = Field(max_length=1000)   # 여러 줄이라 더 길게 준다

summary_schema = SummaryDescription.model_json_schema()

def gen_text_featured_summary(text):
    try:
        messages = []

        messages.append({"role": "user", "content": FEATURED_SUMMARY_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name, messages=messages,
              max_tokens=2048, temperature=0.2, top_p=0.95, seed=1234,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "summary", "schema": summary_schema}},
            )

        # 길이 상한에 걸리면 JSON 이 중간에서 끊긴다. 그런데 API 는 **성공으로**
        # 응답하므로 아래 except 에 걸리지 않고, 몇 셀 뒤에서 json.loads 가
        # "Unterminated string" 으로 죽는다. 원인에서 먼 곳에서 드러나는 것이 가장 나쁘다.
        if completion.choices[0].finish_reason == "length":
            print("[잘림] 출력이 max_tokens 에 걸렸습니다. "
                  "스키마의 max_length 나 max_tokens 를 늘려야 합니다.")
            return 'error'
        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
# user category list를 dict로 변환
describe_by_consumer = {row['user_category']: row['describe']
                        for row in json.loads(user_category_list)['consuber_categories']}

In [ ]:
describe_by_consumer

In [ ]:
# 제품 정보 요약 문장들을 하나로 묶기
feature_summaries = '\n'.join([json.loads(v)['summary'] for k, v in subsection_summary.items()])
print(feature_summaries)

In [ ]:
total_summary_by_consumer = {}

for k, v in describe_by_consumer.items():
    content = f"구매자 유형: {k}\n구매 목적: {v}\n관련 특징 요약:\n{feature_summaries}"
    total_summary_by_consumer[k] = gen_text_featured_summary(content)

In [ ]:
total_summary_by_consumer

### 마지막 — 하나의 파이프라인으로

- 지금까지 한 단계씩 확인한 과정을, 한 번에 실행되는 **하나의 파이프라인**으로 묶습니다.

### 하나씩 하던 것을 전부에 적용합니다

지금까지는 **상품 하나**로 각 단계를 확인했습니다. 이제 **10개 전부**에 돌립니다.

바뀌는 것은 `data.map(...)`으로 감싸는 것뿐입니다. 함수는 그대로입니다.

```
지금까지:  gen_text_step_1(data[0]['text'])        ← 1개
여기부터:  data.map(lambda x: gen_text_step_1(...))  ← 전부
```

**상품 1개에 LLM 호출이 16번쯤** 들어갑니다. 10개면 160번입니다.
그래서 이 셀들이 몇 분 걸립니다.

> 이 숫자를 기억해 두세요. 강사가 배포한 데이터는 **상품 100개**로 만든 것이고,
> 호출이 1,600번 가까이 들어갔습니다. **강의 시간에 할 수 없는 분량**이라
> 미리 만들어 둔 것입니다.
>
> 단계를 쪼개면 결과가 안정되지만 **비용이 그만큼 늡니다.** 그 트레이드오프가
> 여기서 시간으로 드러납니다.

In [ ]:
# Step 1
data = data.map(lambda x: dict(feature_type_list_str=gen_text_step_1(x['text'])))
# Step 2
data = data.map(lambda x: dict(subsection_list_str=gen_text_step_2(x['feature_type_list_str'])))
# Step 3
data = data.map(lambda x: dict(extracted_feature_list_str=
                               gen_text_step_3("Provided Features:\n\n"+ x['feature_type_list_str'] + "\n\n" + x['text'])))
# Step 4
data = data.map(lambda x: dict(user_category_list_str=gen_text_step_4(x['text'])))

In [ ]:
# Step 5

def organize_features(x):
    subsection_dict = {}
    feature_map = json.loads(x['feature_map'])

    subsection_list = json.loads(x['subsection_list_str'])

    for row in subsection_list['subsection_list']:
        subsection = row['subsection']

        subsection_dict[subsection] = {f: feature_map[f]
                                      for f in row['features']
                                      if f in feature_map}

    return dict(subsection_features=json.dumps(subsection_dict))

data = data.map(lambda x: dict(feature_map=json.dumps({row['feature_type']: row['value']
                                                      for row in json.loads(x['extracted_feature_list_str'])['feature_list']})))
data = data.map(organize_features)

In [ ]:
# Step 6

def summarize_subsection(x):
    summary_by_subsection = {}
    subsection_features = json.loads(x['subsection_features'])

    for k, v in subsection_features.items():
        name, features = k, json.dumps(v, indent=4)
        summary_text = gen_text_summary(name + '\n\n' + features)
        summary_by_subsection[name] = summary_text

    return dict(summary_by_subsection=json.dumps(summary_by_subsection))

data = data.map(summarize_subsection)

In [ ]:
def summarize_by_users(x):
    user_category_list = json.loads(x['user_category_list_str'])
    summary_by_subsection = json.loads(x['summary_by_subsection'])

    describe_by_consumer = {row['user_category']: row['describe']
                            for row in user_category_list['consuber_categories']}

    feature_summaries = '\n'.join([json.loads(v)['summary']
                                   for k, v in summary_by_subsection.items()])

    total_summary_by_consumer = {}

    for k, v in describe_by_consumer.items():
        content = f"구매자 유형: {k}\n구매 목적: {v}\n제품 특징 요약:\n{feature_summaries}"
        total_summary_by_consumer[k] = gen_text_featured_summary(content)

    return dict(summary_by_users=json.dumps(total_summary_by_consumer))

data = data.map(summarize_by_users)

In [ ]:
output_data = data.select_columns(['text', 'summary_by_users', 'summary_by_subsection'])

In [ ]:
output_data.to_csv('amazon_gpt_summary.csv')

---

## 산출물 구조 확인

지금까지 7단계를 거쳐 `output_data`를 만들었습니다.
이걸 **다음 실습의 학습 데이터로** 쓰려면 형태를 정확히 알아야 합니다.

특히 `summary_by_users`와 `summary_by_subsection`은 **JSON 문자열 안에 또 JSON 문자열**이
들어 있는 구조입니다. `json.loads`를 두 번 해야 값에 닿습니다.

> 이 셀은 눈으로 확인하기 위한 것입니다. 출력이 예상과 다르면 앞 단계에서
> 무언가 실패한 것입니다 — 다음 단계로 넘어가기 전에 여기서 잡아야 합니다.

In [ ]:
import os
from pathlib import Path


def _repo_root():
    """cwd 에서 위로 올라가며 repo 루트를 찾는다. setup_vessl.sh 를 표지로 쓴다."""
    p = Path.cwd().resolve()
    return next((c for c in [p, *p.parents] if (c / "setup_vessl.sh").exists()), None)


_root = _repo_root()
if "HPC_DATA" in os.environ:
    DATA_DIR = Path(os.environ["HPC_DATA"])          # setup_vessl.sh 가 절대경로로 지정
elif _root is not None:
    DATA_DIR = _root / "data"
else:
    # 조용히 현재 폴더로 폴백하지 않는다.
    # 엉뚱한 곳에 써두면 다음 노트북이 "파일이 없다" 고만 말해서 원인을 못 찾는다.
    raise RuntimeError(
        "데이터 경로를 정할 수 없습니다.\n"
        "  repo 를 통째로 clone 했는지 확인하거나, 터미널에서\n"
        "    export HPC_DATA=/절대/경로\n"
        "  를 지정한 뒤 커널을 재시작하세요."
    )

# assets/ 는 강사가 미리 만들어 배포하는 것(커밋됨), data/ 는 실행하면 생기는 것.
ASSETS_DIR = (_root / "assets") if _root else DATA_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR   = {DATA_DIR}")
print(f"ASSETS_DIR = {ASSETS_DIR}")

In [ ]:
import json

print("컬럼:", output_data.column_names)
print("행 수:", len(output_data))
print()

row = output_data[0]

for col in ["summary_by_subsection", "summary_by_users"]:
    print("=" * 72)
    print(f"[{col}]")
    print("=" * 72)
    v = row[col]
    print(f"  1차 타입: {type(v).__name__}")

    try:
        outer = json.loads(v) if isinstance(v, str) else v
    except (json.JSONDecodeError, TypeError) as e:
        print(f"  ★ 1차 파싱 실패: {type(e).__name__} — 앞 단계가 실패했을 수 있습니다")
        print(f"     원문 앞부분: {str(v)[:200]}")
        continue

    print(f"  1차 파싱 후: {type(outer).__name__}, 키 {list(outer)[:6]}")

    if isinstance(outer, dict) and outer:
        k = next(iter(outer))
        inner_raw = outer[k]
        print(f"  값 타입: {type(inner_raw).__name__}")
        print(f"  값 원문(앞 200자): {str(inner_raw)[:200]}")

        # ★ 여기가 핵심. 안쪽이 또 JSON 문자열인지, 어떤 키를 갖는지.
        try:
            inner = json.loads(inner_raw) if isinstance(inner_raw, str) else inner_raw
            if isinstance(inner, dict):
                print(f"  2차 파싱 성공 → dict, 키 {list(inner)}")
                for ik, iv in list(inner.items())[:2]:
                    print(f"     {ik}: {str(iv)[:150]}")
            else:
                print(f"  2차 파싱 성공 → {type(inner).__name__}: {str(inner)[:150]}")
        except (json.JSONDecodeError, TypeError):
            # 'error' 문자열이 json.dumps 로 감싸이면 파싱은 되고 문자열이 나온다.
            # 여기 걸린다는 것은 애초에 JSON 이 아니라는 뜻이다.
            print("  2차 파싱 불가 — 안쪽은 평문입니다")
    print()

# 실패한 건이 얼마나 되는지 센다. 'error' 는 파싱을 통과하므로 문자열로 직접 본다.
bad = sum(1 for r in output_data
          if "error" in str(r["summary_by_users"])[:40]
          or "error" in str(r["summary_by_subsection"])[:40])
print(f"제외 대상(생성 실패로 보이는 행): {bad} / {len(output_data)}")

---

## 학습 데이터로 바꾸기

지금까지 만든 것은 **요약**입니다. 이걸 내일 파인튜닝(SFT)의 학습 데이터로 씁니다.

SFT는 `instruction`(무엇을 하라)과 `output`(정답) 쌍을 먹습니다.
우리에게는 상품 설명과 요약이 있으니 이렇게 짝지으면 됩니다.

```
instruction : "다음 상품 설명을 읽고 '<구매자 유형>' 관점에서 요약하세요."  + 상품 설명
output      : 그 유형에 대해 만든 요약
```

상품 하나에서 구매자 유형 3~4개 + 속성 그룹 3~5개가 나오므로,
**10개 상품이 수십 건의 학습 예시**가 됩니다.

### 세 가지를 조심해야 합니다

1. **JSON이 두 겹입니다.** 위 구조 확인에서 봤듯이 `json.loads`를 두 번 해야
   `summary`에 닿습니다.
2. **`'error'`는 파싱을 통과합니다.** 실패한 호출이 돌려준 `'error'`가
   `json.dumps`로 감싸이면 `'"error"'`가 되어 파싱은 성공하고 **문자열**이 나옵니다.
   그러면 `obj["summary"]`에서 `KeyError`가 아니라 **`TypeError`**가 납니다.
3. **상품 설명을 잘라야 합니다.** SFT의 `max_length`에 걸리면 **정답이 잘려나가고**
   loss는 낮은데 아무것도 안 배우는 상태가 됩니다. 조용히 실패하는 종류입니다.
4. **원본의 고유명사가 깨져 있습니다.** `- Brand: F, a, t,  , S, h, a, r, k`처럼
   글자가 쉼표로 쪼개져 있습니다. 이대로 학습시키면 모델이 그 패턴을 배웁니다.
   붙여서 복원하고 넣습니다.

In [ ]:
MAX_SRC = 1200   # 상품 설명을 이만큼만 쓴다. 내일 SFT 의 max_length 에 여유를 둔다


def restore_split_names(text):
    """글자가 쉼표로 쪼개진 값을 붙인다.

        - Brand: F, a, t,  , S, h, a, r, k   ->   - Brand: Fat Shark

    원본 데이터의 결함이다. 이대로 학습시키면 모델이
    "브랜드명은 글자를 쉼표로 나눠 쓴다" 를 배운다.

    조각이 3개 이상이고 **전부 한 글자 이하**일 때만 고친다.
    'color, size, weight' 같은 정상 목록은 건드리지 않는다.
    """
    out = []
    for line in text.splitlines(keepends=True):
        head, sep, val = line.partition(":")
        if sep and head.lstrip().startswith("-"):
            parts = [q.strip() for q in val.split(",")]
            if len(parts) >= 3 and all(len(q) <= 1 for q in parts):
                tail = val[len(val.rstrip()):]          # 줄 끝 개행을 보존한다
                line = head + ": " + "".join(q or " " for q in parts).strip() + tail
        out.append(line)
    return "".join(out)


def unwrap(raw):
    """이중 인코딩을 풀어 summary 문자열을 꺼낸다. 못 꺼내면 None."""
    obj = json.loads(raw) if isinstance(raw, str) else raw
    # 'error' 가 감싸이면 obj 가 dict 가 아니라 str 이 된다.
    # 그때 obj["summary"] 는 TypeError 이므로 dict 인지 먼저 본다.
    return obj.get("summary") if isinstance(obj, dict) else None


# 이스케이프를 쓰지 않으려고 삼중따옴표 템플릿으로 둔다.
# (여러 겹의 문자열을 거치면서 백슬래시가 한 겹씩 사라져 실제로 두 번 깨졌다)
INSTRUCTION = """{task}

[상품 설명]
{src}"""

TEMPLATES = [
    ("summary_by_users",
     "다음 상품 설명을 읽고 '{k}' 관점에서 핵심을 요약하세요."),
    ("summary_by_subsection",
     "다음 상품 설명에서 '{k}' 에 해당하는 내용을 한 줄로 요약하세요."),
]

records, dropped = [], 0

for row in output_data:
    src = restore_split_names(row["text"])[:MAX_SRC]
    for col, tmpl in TEMPLATES:
        try:
            outer = json.loads(row[col])
        except (json.JSONDecodeError, TypeError):
            dropped += 1
            continue
        if not isinstance(outer, dict):
            dropped += 1
            continue

        for k, v in outer.items():
            try:
                summary = unwrap(v)
            except (json.JSONDecodeError, TypeError):
                summary = None
            if not summary:
                dropped += 1          # 'error' 나 빈 값
                continue
            records.append({
                "instruction": INSTRUCTION.format(task=tmpl.format(k=k), src=src),
                "output": summary,
            })

# 출력에 'error' 라는 단어를 쓰지 않는다 — 검증 스크립트가 그 단어를 세기 때문이다.
print(f"학습 예시 {len(records)}건 · 제외 {dropped}건")
print(f"  상품 {len(output_data)}개에서 나왔습니다.")

In [ ]:
if records:
    print("=" * 72)
    print("[instruction]")
    print(records[0]["instruction"][:300], "...")
    print()
    print("[output]")
    print(records[0]["output"])
    print("=" * 72)
    print()

    lens = sorted(len(r["instruction"]) + len(r["output"]) for r in records)
    print(f"글자 수 — 중앙값 {lens[len(lens)//2]:,} · 최대 {lens[-1]:,}")
    print("  내일 SFT 에서 토큰 길이 분포를 다시 확인합니다.")

In [ ]:
out_path = DATA_DIR / "amazon_ko_sft.mine.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for r in records:
        # ensure_ascii=False 가 없으면 한글이 escape 되어 눈으로 확인할 수 없다
        print(json.dumps(r, ensure_ascii=False), file=f)

print(f"저장: {out_path}  ({out_path.stat().st_size/1024:.0f} KB)")
print()
print("내일 SFT 실습이 이 파일을 읽습니다.")
print("강사가 미리 만들어 둔 assets/amazon_ko_sft.jsonl.gz 와 합쳐서 학습합니다.")

### 여기서 만든 것이 내일로 이어집니다

| 오늘 만든 것 | 내일 쓰는 곳 |
|---|---|
| `amazon_ko_sft.mine.jsonl` | 3일차 **SFT 학습 데이터** |

수강생이 직접 만든 것은 10개 상품 분량이라 학습에는 적습니다.
그래서 강사가 미리 300개 상품으로 돌려둔 파일과 **합쳐서** 씁니다.
직접 만든 것이 그 안에 들어가 있다는 점이 중요합니다 —
**남의 데이터가 아니라 내가 만든 데이터로 학습**하게 됩니다.

> 이다음 실습(프롬프트 최적화)에서는 학습 없이 프롬프트만으로 점수를 끌어올립니다.
> 내일은 같은 일을 **학습으로** 해보고, 그 차이가 값어치가 있는지 따집니다.